# Jaguar Data Ingestion Pipeline

This notebook demonstrates the complete data ingestion pipeline:
1. Loading data from CSV labels and PPTX files into FiftyOne
2. Video frame sampling
3. Segmentation with SAM3
4. Computing embeddings
5. Exporting processed dataset

Based on `jaguars.ingestion.pipeline`

In [1]:
# Should install the package in editable mode to reflect recent changes
!pip install -e ../

Obtaining file:///sc/home/philipp.kolbe/JID/camera-trap-footage
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jaguars (pyproject.toml) ... done
  Created wheel for jaguars: filename=jaguars-0.1.0-0.editable-py3-none-any.whl size=3285 sha256=8e2402d39b59bad06c3941c6c6042218b7e962fefeb2eb26c565b1dc084c7329
  Stored in directory: /tmp/pip-ephem-wheel-cache-gsxafrtg/wheels/fb/73/df/dfc2bfca6660ef97fbfd8e766c6c3a255e3c7b2656a8649611
Successfully built jaguars
  Attempting uninstall: jaguars
    Found existing installation: jaguars 0.1.0
    Uninstalling jaguars-0.1.0:
      Successfully uninstalled jaguars-0.1.0


## Setup and Imports

In [1]:
import sys
from pathlib import Path
import logging

# Add src to path if needed
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

import fiftyone as fo
from fiftyone import ViewField as F
from jaguars.common.logging_utils import setup_logger
from jaguars.common.config import JID_MASTER_DATASET
from jaguars.ingestion.check_dataset_state import check_dataset_state
from jaguars.ingestion.loaders.csv_loader import ingest_csv_labels
from jaguars.ingestion.loaders.pptx_loader import ingest_pptx_slides
from jaguars.ingestion.processing.sample import run_processing as run_sample
from jaguars.ingestion.processing.add_embeddings import run_processing as run_add_embeddings
from jaguars.ingestion.processing.split import run_processing as run_split
from jaguars.ingestion.processing.deduplicate import run_processing as run_deduplicate
from jaguars.ingestion.export.export import run_processing as run_export

# Setup logging
logger = setup_logger("ingestion_notebook", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Set paths and parameters for the ingestion pipeline.

In [2]:
# Data paths
INPUT_DIR = project_root / "data" / "raw" / "17_11_2025"
LABELS_CSV = INPUT_DIR / "labels.csv"  # Optional: specify exact CSV path
PPTX_PATH = INPUT_DIR / "CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx"

# FiftyOne dataset configuration
DATASET_NAME = JID_MASTER_DATASET  # Default: "JID_Master_Dataset"
PPTX_MEDIA_DIR = None  # Optional: where to save PPTX-extracted images
PPTX_DETECTIONS_FIELD = "pptx_detections"  # Field name for PPTX crop boxes

# CSV ingestion parameters
AUTO_MATCH_MISSING = True  # Auto-match samples that aren't in CSV
MATCH_THRESHOLD = 0.95  # Similarity threshold for auto-matching
SUGGEST_THRESHOLD = 0.80  # Similarity threshold for suggestions

# Processing parameters
SEGMENTATION_FIELD = "sam3_segmentations"
SEGMENTATION_PROMPT = "jaguar"

# Split parameters
RUN_SPLIT = True
TAG_SPLITS = True  # Adds train/val/test tags from closed_set_split

# Deduplication parameters
RUN_DEDUP = True
DEDUP_SIMILARITY_THRESHOLD = 0.98

# Export configuration
EXPORT_BASE_DIR = project_root / "data" / "intermediate" / "v1" / "fo_jaguars" / "exports"
# Updated variants to match the new export system
EXPORT_VARIANTS = [
    "master",                        # Full dataset with all samples
    "segmented_deduplicated",        # Segmented samples, duplicates removed
    "segmented",                     # Segmented samples, including duplicates  
    "not_segmented_deduplicated",    # Non-segmented samples, duplicates removed
    "not_segmented",                 # Non-segmented samples, including duplicates
]
EXPORT_TARGETS = ["disk"]  # Options: "disk", "fiftyone", "huggingface"
HUGGINGFACE_REPO = None  # e.g. "jid-ingestion"

# Runtime
OVERWRITE_DATASET = True  # Set to True to delete existing dataset
VERBOSE = True

print("Configuration:")
print(f"  Input directory: {INPUT_DIR}")
print(f"  Labels CSV: {LABELS_CSV if LABELS_CSV else 'Auto-detect in input_dir'}")
print(f"  PPTX file: {PPTX_PATH}")
print(f"  Dataset name: {DATASET_NAME}")
print(f"  Export base dir: {EXPORT_BASE_DIR}")
print(f"  Export variants: {len(EXPORT_VARIANTS)} variants")
for variant in EXPORT_VARIANTS:
    print(f"    - {variant}")
print(f"  Export targets: {EXPORT_TARGETS}")
print(f"  Overwrite dataset: {OVERWRITE_DATASET}")

Configuration:
  Input directory: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025
  Labels CSV: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/labels.csv
  PPTX file: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx
  Dataset name: JID_Master_Dataset
  Export base dir: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports
  Export variants: ['full', 'deduplicated', 'segmented', 'segmented_deduplicated']
  Export targets: ['disk']
  Overwrite dataset: True


In [4]:
check_dataset_state(DATASET_NAME)

✓ Dataset 'JID_Master_Dataset' exists

DATASET OVERVIEW
Total samples: 1171
Group field: group
Image samples: 1171
Video samples: 230

FIELDS
  id: fiftyone.core.fields.ObjectIdField
  filepath: fiftyone.core.fields.StringField
  tags: fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
  metadata: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
  created_at: fiftyone.core.fields.DateTimeField
  last_modified_at: fiftyone.core.fields.DateTimeField
  group: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
  source_type: fiftyone.core.fields.StringField
  source: fiftyone.core.fields.StringField
  csv_source: fiftyone.core.fields.StringField
  jaguar_id: fiftyone.core.fields.StringField
  ground_truth: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
  site: fiftyone.core.fields.StringField
  cam: fiftyone.core.fields.StringField
  sighting_id: fiftyone.core.fields.StringField
  date: fiftyone.c

{'exists': True,
 'images_count': 1171,
 'videos_count': 230,
 'csv_loaded': True,
 'pptx_loaded': True,
 'frames_sampled': 1030,
 'segmentation_sam3_segmentations': 1171,
 'detection_embeddings_sam3_segmentations': 1171}

In [3]:
# Remove existing dataset if needed
if OVERWRITE_DATASET and fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
    print(f"✓ Deleted existing dataset '{DATASET_NAME}'")

In [5]:
if fo.dataset_exists(DATASET_NAME):
    dataset = fo.load_dataset(DATASET_NAME)
    print(f"✓ Loaded existing dataset '{DATASET_NAME}' with {len(dataset)} samples")

✓ Loaded existing dataset 'JID_Master_Dataset' with 1171 samples


## Step 1: Data Loading

Load data from CSV labels and PPTX files into FiftyOne.

### Step 1a: CSV Labels Ingestion

Ingest video metadata and labels from CSV file.

In [5]:
print("=" * 70)
print("STEP 1a: CSV Labels Ingestion")
print("=" * 70)

# Ingest CSV labels
csv_kwargs = {
    "input_dir": INPUT_DIR,
    "dataset_name": DATASET_NAME,
    "auto_match_missing": AUTO_MATCH_MISSING,
    "match_threshold": MATCH_THRESHOLD,
    "suggest_threshold": SUGGEST_THRESHOLD,
}

if LABELS_CSV and LABELS_CSV.exists():
    csv_kwargs["input_csv"] = LABELS_CSV

print(f"Loading from: {INPUT_DIR}")
dataset = ingest_csv_labels(**csv_kwargs)

print("\n✓ CSV ingestion completed!")
print(f"  Dataset: {dataset.name}")
print(f"  Total samples: {len(dataset)}")
print(f"  Group field: {dataset.group_field}")

# Access image and video slices
images_view = dataset.select_group_slices("image")
videos_view = dataset.select_group_slices("video")

print(f"  Image samples: {len(images_view)}")
print(f"  Video samples: {len(videos_view)}")

STEP 1a: CSV Labels Ingestion
Loading from: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - Cleaning labels using CSV cleaning pipeline: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/labels.csv
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - Loaded 241 rows with columns: CAMERA TRAP SITE , LATITUDE, LONGITUDE, CAMERA ID, CAM , JAGUAR ID, LOCATION, CAMERA MODEL, DATE, TIME, TEMP C, Files Name, NOTES/ ERRORS, Gaia lat, Gaia long
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - All optional columns present
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - Filling 4 missing JAGUAR IDs
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - Assigned 4 new JAGUAR IDs
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - Parsed 241/241 DATE entries
11:40:59 - jid_logger.ingestion.loaders.csv_loader - INFO - Split 24 rows with multifile sightings into 2

Converting AVI to MP4: 100%|██████████| 14/14 [12:15<00:00, 52.53s/it]


11:53:16 - jid_logger.ingestion.loaders.csv_loader - INFO - Found 30 video files without labels


INFO:jid_logger.ingestion.loaders.csv_loader:Found 30 video files without labels


11:53:16 - jid_logger.common.fiftyone_utils - INFO - Creating new dataset: JID_Master_Dataset


...

11:53:16 - jid_logger.ingestion.loaders.csv_loader - INFO - Added 251 samples to grouped dataset (21 images, 230 videos)


INFO:jid_logger.ingestion.loaders.csv_loader:Added 251 samples to grouped dataset (21 images, 230 videos)


11:53:16 - jid_logger.ingestion.loaders.csv_loader - INFO - CSV Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "is_grouped": true,
  "slices": [
    "image",
    "video"
  ],
  "csv_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/labels.csv",
  "new_image_samples_added": 21,
  "new_video_samples_added": 230,
  "total_image_samples_in_dataset": 21,
  "total_video_samples_in_dataset": 230,
  "total_samples_in_dataset": 251,
  "cleaned_rows_total": 271
}


INFO:jid_logger.ingestion.loaders.csv_loader:CSV Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "is_grouped": true,
  "slices": [
    "image",
    "video"
  ],
  "csv_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/labels.csv",
  "new_image_samples_added": 21,
  "new_video_samples_added": 230,
  "total_image_samples_in_dataset": 21,
  "total_video_samples_in_dataset": 230,
  "total_samples_in_dataset": 251,
  "cleaned_rows_total": 271
}



✓ CSV ingestion completed!
  Dataset: JID_Master_Dataset
  Total samples: 21
  Group field: group
  Image samples: 21
  Video samples: 230


### Step 1b: PPTX Ingestion

Extract and ingest jaguar reference images from PowerPoint presentation.

In [6]:
print("=" * 70)
print("STEP 1b: PPTX Ingestion")
print("=" * 70)

if PPTX_PATH and PPTX_PATH.exists():
    print(f"Loading PPTX: {PPTX_PATH}")
    
    dataset = ingest_pptx_slides(
        pptx_path=PPTX_PATH,
        dataset_name=DATASET_NAME,
        media_dir=PPTX_MEDIA_DIR,
        detections_field=PPTX_DETECTIONS_FIELD,
    )
    
    print("\n✓ PPTX ingestion completed!")
    
    # Refresh views
    images_view = dataset.select_group_slices("image")
    videos_view = dataset.select_group_slices("video")
    
    print(f"  Total samples: {len(dataset)}")
    print(f"  Image samples: {len(images_view)}")
    print(f"  Video samples: {len(videos_view)}")
    
    # Check for PPTX-derived samples
    pptx_samples = images_view.match_tags("pptx")
    print(f"  PPTX-derived samples: {len(pptx_samples)}")
else:
    print(f"⚠ PPTX file not found: {PPTX_PATH}")
    print("Skipping PPTX ingestion")

STEP 1b: PPTX Ingestion
Loading PPTX: /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx
11:53:51 - jid_logger.ingestion.loaders.pptx_loader - INFO - Loading PowerPoint from /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx


INFO:jid_logger.ingestion.loaders.pptx_loader:Loading PowerPoint from /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx


11:53:51 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset


INFO:jid_logger.common.fiftyone_utils:Loading existing dataset: JID_Master_Dataset


11:53:51 - jid_logger.ingestion.loaders.pptx_loader - INFO - Loading PowerPoint from /sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx


...

11:53:53 - jid_logger.ingestion.loaders.pptx_loader - INFO - Added 135 samples from PPTX


INFO:jid_logger.ingestion.loaders.pptx_loader:Added 135 samples from PPTX


11:53:53 - jid_logger.ingestion.loaders.pptx_loader - INFO - PPTX Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "pptx_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx",
  "media_dir": "/sc/home/philipp.kolbe/fiftyone/JID_Master_Dataset/media/pptx",
  "slides_total": 80,
  "slides_with_images": 77,
  "images_extracted": 135,
  "new_samples_added": 135,
  "total_samples_in_dataset": 156,
  "unique_jaguar_ids": 69,
  "unique_sites": 10
}


INFO:jid_logger.ingestion.loaders.pptx_loader:PPTX Ingestion Summary: {
  "dataset_name": "JID_Master_Dataset",
  "pptx_source": "/sc/home/philipp.kolbe/JID/camera-trap-footage/data/raw/17_11_2025/CAMERA TRAP ID GUIDE UPDATED by Oscar2025.pptx",
  "media_dir": "/sc/home/philipp.kolbe/fiftyone/JID_Master_Dataset/media/pptx",
  "slides_total": 80,
  "slides_with_images": 77,
  "images_extracted": 135,
  "new_samples_added": 135,
  "total_samples_in_dataset": 156,
  "unique_jaguar_ids": 69,
  "unique_sites": 10
}



✓ PPTX ingestion completed!
  Total samples: 156
  Image samples: 156
  Video samples: 230
  PPTX-derived samples: 0


### Visualize Loaded Dataset

Launch FiftyOne App to inspect the ingested data.

In [ ]:
# Launch FiftyOne App (optional)
session = fo.launch_app(dataset)
print(f"\n✓ FiftyOne App launched for dataset: {dataset.name}")
print("Explore the data in your browser!")

## Step 2: Video Frame Sampling

Extract frames from videos at regular intervals.

In [7]:
print("=" * 70)
print("STEP 2: Video Frame Sampling")
print("=" * 70)

sampling_results = run_sample(
    dataset_name=DATASET_NAME,
    verbose=VERBOSE
)

STEP 2: Video Frame Sampling
11:54:11 - jid_logger.ingestion.processing.sample - INFO - Starting video frame sampling for dataset: JID_Master_Dataset


INFO:jid_logger.ingestion.processing.sample:Starting video frame sampling for dataset: JID_Master_Dataset


11:54:11 - jid_logger.ingestion.processing.sample - INFO - Sampling strategy: 1.0 fps for first 5.0 seconds, then 0.16666666666666666 fps for remainder


INFO:jid_logger.ingestion.processing.sample:Sampling strategy: 1.0 fps for first 5.0 seconds, then 0.16666666666666666 fps for remainder


11:54:11 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset


...

INFO:jid_logger.ingestion.processing.sample:Added 2298 sampled frames from 230 videos


12:06:38 - jid_logger.ingestion.processing.sample - INFO - Sampling Summary: {
  "videos_processed": 230,
  "frames_added": 2298,
  "total_image_samples": 2454,
  "total_video_samples": 230,
  "total_samples": 2454
}


INFO:jid_logger.ingestion.processing.sample:Sampling Summary: {
  "videos_processed": 230,
  "frames_added": 2298,
  "total_image_samples": 2454,
  "total_video_samples": 230,
  "total_samples": 2454
}


12:06:38 - jid_logger.ingestion.processing.sample - INFO - Video frame sampling completed successfully.


INFO:jid_logger.ingestion.processing.sample:Video frame sampling completed successfully.


In [8]:
# Refresh dataset
dataset.reload()
images_view = dataset.select_group_slices("image")
print(f"  Total image samples (after sampling): {len(images_view)}")

  Total image samples (after sampling): 2454


## Step 3: Segmentation with SAM3

Detect and segment jaguars in images using SAM3.

In [9]:
print("=" * 70)
print("STEP 3: Segmentation & Filtering")
print("=" * 70)

# Import segmentation processing
try:
    from jaguars.ingestion.processing.segmentation import run_processing as run_segmentation
    
    segmentation_results = run_segmentation(
        dataset_name=DATASET_NAME,
        segmentation_field=SEGMENTATION_FIELD,
        prompt=SEGMENTATION_PROMPT,
        inspect_app=False,  # Set to True to inspect results
        verbose=VERBOSE
    )
    
    print("\n✓ Segmentation completed!")    
    # Refresh dataset
    dataset.reload()
    
except ImportError as e:
    print(f"⚠ Segmentation module not available: {e}")
    print("Skipping segmentation step")
    segmentation_results = None

STEP 3: Segmentation & Filtering
12:17:22 - jid_logger.ingestion.processing.segmentation - INFO - Starting segmentation processing pipeline for: JID_Master_Dataset


INFO:jid_logger.ingestion.processing.segmentation:Starting segmentation processing pipeline for: JID_Master_Dataset


12:17:22 - jid_logger.ingestion.processing.segmentation - INFO - Running SAM3...


INFO:jid_logger.ingestion.processing.segmentation:Running SAM3...


12:17:22 - jid_logger.segmentation.SAM3 - INFO - Starting SAM3 segmentation for dataset: JID_Master_Dataset


...

12:42:32 - jid_logger.ingestion.processing.segmentation - INFO - Deleting 1283 samples with tags: ['filter_count', 'filter_quality']


INFO:jid_logger.ingestion.processing.segmentation:Deleting 1283 samples with tags: ['filter_count', 'filter_quality']


12:42:32 - jid_logger.ingestion.processing.segmentation - INFO - Segmentation pipeline complete.


INFO:jid_logger.ingestion.processing.segmentation:Segmentation pipeline complete.



✓ Segmentation completed!


## Step 4: Compute Embeddings

Generate embeddings for jaguar detections using a pre-trained model.

In [10]:
print("=" * 70)
print("STEP 4: Embedding Computation")
print("=" * 70)

mask_embedding_results = run_add_embeddings(
    dataset_name=DATASET_NAME,
    patches_field=SEGMENTATION_FIELD,  # Use SAM3 segmentations
    mask_field="mask",  # SAM3 adds mask field
    verbose=VERBOSE
)

print("\n✓ Segmented Mask Embedding computation completed!")

full_image_embedding_results = run_add_embeddings(
    dataset_name=DATASET_NAME,
    verbose=VERBOSE
)

print("\n✓ Full image embedding computation completed!")

# Refresh dataset
dataset.reload()

STEP 4: Embedding Computation
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Starting embedding computation for dataset: JID_Master_Dataset
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Model: hf-hub:BVRA/MegaDescriptor-L-384
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Target Field: embeddings_BVRA_MegaDescriptor_L_384
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Processing patches from field: sam3_segmentations
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Using masks from field: mask
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Batch Size: 32
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Selected 'image' slice for processing. 1171 samples found.
14:02:44 - jid_logger.ingestion.processing.add_embeddings - INFO - Gathering detection patches from field 'sam3_segmentations'...
14:02:58 - jid_logger.ingestion.processing.add_embeddings - INFO

14:03:03 - jid_logger.ingestion.processing.add_embeddings - INFO - Model loaded in 4.95 seconds
14:03:03 - jid_logger.ingestion.processing.add_embeddings - INFO - Computing embeddings...


14:04:20 - jid_logger.ingestion.processing.add_embeddings - INFO - Computed 1171 embeddings in 77.11 seconds
14:04:20 - jid_logger.ingestion.processing.add_embeddings - INFO - Saving embeddings to field 'embeddings_BVRA_MegaDescriptor_L_384'...
14:04:20 - jid_logger.ingestion.processing.add_embeddings - INFO - Updating 1171 samples with new detection embeddings...


14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Embedding computation completed successfully.

✓ Embedding computation completed!


In [11]:
full_image_embedding_results = run_add_embeddings(
    dataset_name=DATASET_NAME,
    verbose=VERBOSE
)

print("\n✓ Full image embedding computation completed!")

# Refresh dataset
dataset.reload()

14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Starting embedding computation for dataset: JID_Master_Dataset
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Model: hf-hub:BVRA/MegaDescriptor-L-384
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Target Field: embeddings_BVRA_MegaDescriptor_L_384
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Batch Size: 32
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Selected 'image' slice for processing. 1171 samples found.
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Gathering inputs...
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Using device: cuda
14:05:00 - jid_logger.ingestion.processing.add_embeddings - INFO - Loading model hf-hub:BVRA/MegaDescriptor-L-384...
14:05:04 - jid_logger.ingestion.processing.add_embeddings - INFO - Model loaded in 3.81 seconds
14:05:04 - jid_logger.ingestion.processing.add_embe

14:06:16 - jid_logger.ingestion.processing.add_embeddings - INFO - Computed 1171 embeddings in 72.24 seconds
14:06:16 - jid_logger.ingestion.processing.add_embeddings - INFO - Saving embeddings to field 'embeddings_BVRA_MegaDescriptor_L_384'...


14:06:17 - jid_logger.ingestion.processing.add_embeddings - INFO - Embedding computation completed successfully.

✓ Full image embedding computation completed!


## Step 5: Split Dataset

Create train/val/test splits and optionally tag samples.

In [13]:
print("=" * 70)
print("STEP 5: Dataset Splitting")
print("=" * 70)

if RUN_SPLIT:
    split_results = run_split(
        dataset_name=DATASET_NAME,
        add_closed_set=True,
        add_open_set=False,
        tag_by="closed" if TAG_SPLITS else None,
        verbose=VERBOSE,
    )
    print("\n✓ Split completed!")
else:
    print("Skipping split step")
    split_results = None

STEP 5: Dataset Splitting
14:08:50 - jid_logger.ingestion.processing.split - INFO - Starting split processing for dataset: JID_Master_Dataset
14:08:50 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset
14:08:50 - jid_logger.ingestion.processing.split - INFO - Filtering dataset to 'image' group slice only
14:08:50 - jid_logger.ingestion.processing.split - INFO - Working with 1171 samples from 'image' slice
14:08:50 - jid_logger.ingestion.processing.split - INFO - Using ID field: jaguar_id
14:08:50 - jid_logger.ingestion.processing.split - INFO - Applying Closed-Set Split...


Closed-set split: 100%|██████████| 76/76 [00:00<00:00, 284.95it/s]


14:08:51 - jid_logger.ingestion.processing.split - INFO - Verifying Closed-Set integrity...
14:08:51 - jid_logger.ingestion.processing.split - INFO - Counts for closed_set_split: {'val': 120, 'test': 105, 'train': 946}
14:08:51 - jid_logger.ingestion.processing.split - INFO - Tagging samples by closed-set split...
14:08:52 - jid_logger.ingestion.processing.split - INFO - Tagged 946 samples with 'train'
14:08:52 - jid_logger.ingestion.processing.split - INFO - Tagged 120 samples with 'val'
14:08:52 - jid_logger.ingestion.processing.split - INFO - Tagged 105 samples with 'test'

✓ Split completed!


## Step 6: Deduplicate Images

Flag near-duplicate images (no segmentation, full image only).

In [20]:
# reload run_deduplicate package so we dont have to restart
import importlib
import jaguars.ingestion.processing.deduplicate as dedup_module
importlib.reload(dedup_module)

from jaguars.ingestion.processing.deduplicate import run_processing as run_deduplicate

In [21]:
print("=" * 70)
print("STEP 6: Deduplication")
print("=" * 70)

if RUN_DEDUP:
    run_deduplicate(
        dataset_name=DATASET_NAME,
        similarity_threshold=DEDUP_SIMILARITY_THRESHOLD,
        verbose=VERBOSE,
    )
    print("\n✓ Deduplication completed!")
else:
    print("Skipping deduplication step")

STEP 6: Deduplication
14:18:26 - jid_logger.ingestion.processing.deduplicate - INFO - Starting deduplication for dataset: JID_Master_Dataset
14:18:26 - jid_logger.common.fiftyone_utils - INFO - Loading existing dataset: JID_Master_Dataset
14:18:26 - jid_logger.ingestion.processing.deduplicate - INFO - Clearing previous duplicate markings...
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Previous duplicate markings cleared
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Using embeddings field: embeddings_BVRA_MegaDescriptor_L_384
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Embeddings field 'embeddings_BVRA_MegaDescriptor_L_384' already exists, using existing embeddings
14:18:27 - jid_logger.ingestion.processing.deduplicate - INFO - Running FiftyOne brain near duplicate detection (threshold=0.980)
14:18:28 - jid_logger.ingestion.processing.deduplicate - DEBUG - Marked 69808241341041adb61e5fe3 as duplicate of 6980821c341041adb61e5d89 (s

Name:        JID_Master_Dataset
Media type:  group
Group slice: image
Num groups:  1171
Persistent:  True
Tags:        []
Sample fields:
    id:                                   fiftyone.core.fields.ObjectIdField
    filepath:                             fiftyone.core.fields.StringField
    tags:                                 fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                             fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
    created_at:                           fiftyone.core.fields.DateTimeField
    last_modified_at:                     fiftyone.core.fields.DateTimeField
    group:                                fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
    source_type:                          fiftyone.core.fields.StringField
    source:                               fiftyone.core.fields.StringField
    csv_source:                           fiftyone.core.fields.String

## Step 7: Export Dataset Variants

Export different dataset variants to disk, FiftyOne, or Huggingface.

In [22]:
print("=" * 70)
print("STEP 7: Export Dataset Variants")
print("=" * 70)

export_results = run_export(
    dataset_name=DATASET_NAME,
    variants=EXPORT_VARIANTS,
    export_targets=EXPORT_TARGETS,
    huggingface_repo=HUGGINGFACE_REPO,
    export_base_dir=EXPORT_BASE_DIR,
    segmentation_field=SEGMENTATION_FIELD,
    dedup_field="is_duplicate",
    verbose=VERBOSE,
)

print("\n✓ Export completed!")
print(f"  Base export dir: {EXPORT_BASE_DIR}")

STEP 7: Export Dataset Variants
14:57:52 - jid_logger.ingestion.export.export - INFO - Starting export for dataset: JID_Master_Dataset
14:57:52 - jid_logger.ingestion.export.export - INFO - Exporting 'full' to disk at /sc/home/philipp.kolbe/JID/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports/full
Exporting samples...
 100% |██████████████████| 1401/1401 [1.3m elapsed, 0s remaining, 118.2 docs/s]      
Exporting frames...
 100% |████████████████████████| 0/0 [303.6us elapsed, ? remaining, ? docs/s] 
14:59:12 - jid_logger.ingestion.export.export - INFO - Exporting 'deduplicated' to disk at /sc/home/philipp.kolbe/JID/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports/deduplicated
Exporting samples...
 100% |██████████████████| 1065/1065 [11.7s elapsed, 0s remaining, 104.6 docs/s]      
14:59:23 - jid_logger.ingestion.export.export - INFO - Exporting 'segmented' to disk at /sc/home/philipp.kolbe/JID/camera-trap-footage/data/intermediate/v1/fo_jaguars/exports/segmented


## Pipeline Summary

Display overall statistics and results from the ingestion pipeline.

In [ ]:
check_dataset_state(DATASET_NAME)

## Next Steps

The ingestion pipeline is complete! For detailed dataset analysis and exploration, see the `dataset_analysis.ipynb` notebook.